# Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import json
import requests
from pathlib import Path
from typing import Dict, List, Optional
import warnings
warnings.filterwarnings('ignore')

# Configuration

In [2]:
pwd

'c:\\Users\\praka\\Acads\\Work\\AI_Agents\\linkedn-scrapper-v2\\linkedin_scraper_repo\\samples'

In [3]:
# Configuration
excel_file_path = "DivisionWiseRollList201723July.xlsx"  # Path to Excel file with multiple sheets
output_csv = "input_people_csv_v3.csv"
college_name = "Indian Institute of Technology, Bombay"
llm_model = "qwen3:4b"  # Using Ollama's mistral model (lightweight)
ollama_api_url = "http://localhost:11434/api/generate"  # Ollama API endpoint
use_ollama = True  # Set to False if you want to disable LLM parsing

print("Configuration:")
print(f"  Excel file: {excel_file_path}")
print(f"  Output CSV: {output_csv}")
print(f"  College name: {college_name}")
print(f"  LLM Model: {llm_model}")
print(f"  Using Ollama: {use_ollama}")

Configuration:
  Excel file: DivisionWiseRollList201723July.xlsx
  Output CSV: input_people_csv_v3.csv
  College name: Indian Institute of Technology, Bombay
  LLM Model: qwen3:4b
  Using Ollama: True


# Test Ollama Connectivity (Early Check)

In [4]:
import ollama

# Check the currently running models (optional)
print(ollama.ps())

# Send a query to the model
response = ollama.chat(model=llm_model, messages=[
    {'role': 'user', 'content': 'Explain the difference between a list and a tuple in Python.'},
])

print(response['message']['content'])


models=[Model(model='deepseek-coder-v2:latest', name='deepseek-coder-v2:latest', digest='63fb193b3a9b4322a18e8c6b250ca2e70a5ff531e962dbf95ba089b2566f2fa5', expires_at=datetime.datetime(2026, 1, 23, 0, 27, 0, 590353, tzinfo=TzInfo(19800)), size=11458461696, size_vram=5783670784, details=ModelDetails(parent_model='', format='gguf', family='deepseek2', families=['deepseek2'], parameter_size='15.7B', quantization_level='Q4_0'), context_length=8192)]
Here's a clear, practical explanation of the key differences between **lists** and **tuples** in Python, with examples and when to use each:

---

### 🌟 Core Difference: **Mutability**
| Feature          | List (`[]`)                          | Tuple (`()`)                          |
|-------------------|---------------------------------------|----------------------------------------|
| **Mutable?**      | ✅ **Yes** (can be changed)          | ❌ **No** (immutable, cannot be changed) |
| **Why?**          | Stores references to objects (dynamic)

# Load Excel File and Read All Sheets

In [5]:
# Check if file exists
if not Path(excel_file_path).exists():
    print(f"❌ File not found: {excel_file_path}")
    print(f"Please provide the correct path to your Excel file.")
else:
    # Read all sheet names from Excel file
    xls = pd.ExcelFile(excel_file_path)
    sheet_names = xls.sheet_names
    print(f"✓ Found Excel file: {excel_file_path}")
    print(f"✓ Total sheets: {len(sheet_names)}")
    print(f"\nSheet names:")
    for i, sheet in enumerate(sheet_names, 1):
        print(f"  {i}. {sheet}")
    
    # Load all sheets: read BOTH header row (row 0) and data rows
    sheets_data = {}
    sheets_headers = {}  # Store the first row (with dept info) separately
    
    for sheet_name in sheet_names:
        # First, read the FIRST ROW (row 0) which contains department info
        df_header = pd.read_excel(excel_file_path, sheet_name=sheet_name, header=None, nrows=1)
        header_text = " ".join([str(x) for x in df_header.iloc[0].values if pd.notna(x)])
        sheets_headers[sheet_name] = header_text
        
        # Then, read the DATA ROWS (skip first row which is the header)
        df_data = pd.read_excel(excel_file_path, sheet_name=sheet_name, skiprows=1, header=None)
        sheets_data[sheet_name] = df_data
        
        print(f"\n  Sheet '{sheet_name}': {len(df_data)} data rows")
        print(f"    Header/Department info: {header_text[:80]}...")
        if len(df_data) > 0:
            print(f"    First student row: {df_data.iloc[0, :3].tolist()}")
        else:
            print(f"    No student data in this sheet")

✓ Found Excel file: DivisionWiseRollList201723July.xlsx
✓ Total sheets: 15

Sheet names:
  1. Table 1
  2. Table 2
  3. Table 3
  4. Table 4
  5. Table 5
  6. Table 6
  7. Table 7
  8. Table 8
  9. Table 9
  10. Table 10
  11. Table 11
  12. Table 12
  13. Table 13
  14. Table 14
  15. Table 15

  Sheet 'Table 1': 63 data rows
    Header/Department info: Program : B.Tech.
Academic Year-Semester 2017-1 Department : Aerospace Engineeri...
    First student row: [np.float64(1.0), np.float64(170010001.0), 'Meena Sourav Ramkishor']

  Sheet 'Table 2': 124 data rows
    Header/Department info: Program : B.Tech.
Academic Year-Semester 2017-1 Department : Chemical Engineerin...
    First student row: [np.int64(1), np.int64(170020001), 'Pradyum Soni']

  Sheet 'Table 3': 118 data rows
    Header/Department info: Program : B.Tech.
Academic Year-Semester 2017-1 Department : Civil Engineering B...
    First student row: [1, np.float64(170040002.0), 'Ameya Shankar Kulkarni']

  Sheet 'Table 4': 111

# Initialize LLM for Department Parsing

In [6]:
def parse_department_with_llm(sheet_name: str, first_row_text: str):
    """
    Use Ollama LLM to parse sheet name/header and extract department name.
    Falls back to simple heuristic if LLM is not available.
    """

    try:
        # Create prompt for LLM
        prompt = f"""Extract the department name from the following text. 
        Return ONLY the department name, nothing else.
        
        Sheet name: {sheet_name}
        Header text: {first_row_text}
        
        Look for **department** key in the text provided, find the department name and return it.

        FEW SHOT EXAMPLES:
        -----------------
        Example 1:
        Sheet name: table1
        Header text: " Program : B.Tech.Academic Year-Semester 2017-1 Department : Aerospace Engineering Batch Year : 2017..."

        Output: Aerospace Engineering
        -----------------

        SPECIAL INSTRUCTIONS:
        - DO NOT return anything other than the department name.
        - DO NOT return any explanations, logic, or extra text.
        - your output MUST be only the department name (e.g., "Computer Science and Engineering", "Electrical Engineering", etc.)


        Output format:
        Department name:

        """
        
        # Call Ollama API
        # print('final prompt to model:', prompt)
        response = ollama.chat(model=llm_model, 
                               messages=[{'role': 'user', 'content': prompt},])

        print(f"LLM response for '{sheet_name}': {response['message']['content']}")
        
        # if response.status_code == 200:
        #     result = response.json()
        #     department = result.get("response", "").strip()
        #     if department:
        #         return department
        department = response['message']['content'].strip()
        if department:
            return department
        
    except Exception as e:
        print(f"  ⚠ LLM error for '{sheet_name}': {str(e)}")
    
    # Fallback to heuristic
    return extract_department_heuristic(sheet_name, first_row_text)

def extract_department_heuristic(sheet_name: str, first_row_text: str) -> str:
    """
    Simple heuristic to extract department name from sheet name or text.
    Common IIT departments mapping.
    """
    common_depts = {
        'CSE': 'Computer Science and Engineering',
        'ECE': 'Electrical Engineering',
        'ME': 'Mechanical Engineering',
        'CE': 'Civil Engineering',
        'MET': 'Metallurgical Engineering',
        'AE': 'Aerospace Engineering',
        'CHE': 'Chemical Engineering',
        'BT': 'Biotechnology',
        'DD': 'Dual Degree',
    }
    
    # Check if sheet name contains dept abbreviation
    for abbr, full_name in common_depts.items():
        if abbr in sheet_name.upper():
            return full_name
    
    # Try to find department from first_row_text
    combined_text = sheet_name + " " + str(first_row_text).upper()
    for abbr, full_name in common_depts.items():
        if abbr in combined_text:
            return full_name
    
    # Default: return cleaned sheet name
    return sheet_name.strip()

print("✓ LLM parsing functions initialized")

✓ LLM parsing functions initialized


# Extract Student Data from All Sheets

In [7]:
# Extract roll numbers and names from each sheet
# Columns: 2 (roll number, index 1) and 3 (name, index 2)

extracted_data = []

for sheet_name, df in sheets_data.items():
    print(f"\n📋 Processing sheet: '{sheet_name}'")
    
    # Extract columns 2 and 3 (0-indexed: 1 and 2)
    if len(df.columns) < 3:
        print(f"  ⚠ Sheet has fewer than 3 columns, skipping")
        continue
    
    roll_numbers = df.iloc[:, 1]  # Column 2 (index 1)
    names = df.iloc[:, 2]         # Column 3 (index 2)
    
    # Remove NaN values and empty rows
    valid_rows = []
    for idx in range(len(roll_numbers)):
        roll = roll_numbers.iloc[idx]
        name = names.iloc[idx]
        
        # Skip if roll number or name is NaN or empty
        if pd.isna(roll) or pd.isna(name):
            continue
        if str(roll).strip() == '' or str(name).strip() == '':
            continue
        
        valid_rows.append({
            'roll_number': str(roll).strip(),
            'name': str(name).strip(),
            'sheet_name': sheet_name
        })
    
    print(f"  ✓ Extracted {len(valid_rows)} valid student records")
    if len(valid_rows) > 0:
        print(f"    Sample: Roll={valid_rows[0]['roll_number']}, Name={valid_rows[0]['name']}")
    
    extracted_data.extend(valid_rows)

print(f"\n✓ Total students extracted from all sheets: {len(extracted_data)}")

# Create intermediate dataframe
temp_df = pd.DataFrame(extracted_data)
if len(temp_df) > 0:
    print(f"\nExtracted data summary:")
    print(f"  Unique sheets: {temp_df['sheet_name'].nunique()}")
    print(f"  Total records: {len(temp_df)}")
    print(f"\nRecords per sheet:")
    for sheet, count in temp_df['sheet_name'].value_counts().items():
        print(f"  {sheet}: {count} students")


📋 Processing sheet: 'Table 1'
  ✓ Extracted 62 valid student records
    Sample: Roll=170010001.0, Name=Meena Sourav Ramkishor

📋 Processing sheet: 'Table 2'
  ✓ Extracted 124 valid student records
    Sample: Roll=170020001, Name=Pradyum Soni

📋 Processing sheet: 'Table 3'
  ✓ Extracted 117 valid student records
    Sample: Roll=170040002.0, Name=Ameya Shankar Kulkarni

📋 Processing sheet: 'Table 4'
  ✓ Extracted 111 valid student records
    Sample: Roll=170050001, Name=Piyush Dilip Tibarewal

📋 Processing sheet: 'Table 5'
  ✓ Extracted 60 valid student records
    Sample: Roll=170070001, Name=Arvadia Kevin Dharmesh

📋 Processing sheet: 'Table 6'
  ✓ Extracted 116 valid student records
    Sample: Roll=170100001, Name=Kshitiz Singhal

📋 Processing sheet: 'Table 7'
  ✓ Extracted 98 valid student records
    Sample: Roll=170110001, Name=Milind Vijay Chandnani

📋 Processing sheet: 'Table 8'
  ✓ Extracted 42 valid student records
    Sample: Roll=170260001, Name=Jigyasu Chand

📋 Process

# Parse Department Names Using LLM

In [8]:
import re

def parse_department_with_regexp(sheet_name, first_row_text):
    # This pattern looks for "Department :" literal, captures everything (.*?) until "Batch"
    # The (.*?) is a "non-greedy" match, stopping at the first "Batch" it finds.
    pattern = r"Department :(.*?)Batch"
    
    match = re.search(pattern, first_row_text)
    
    if match:
        # match.group(1) is the text inside the parentheses
        # .strip() removes the leading and trailing spaces
        return match.group(1).strip()
    
    return None  # Or return "" if you prefer an empty string on failure

# --- Usage Example ---
first_row_text = "Generated Report Department : Computer Science & Engineering  Batch 2024"
sheet_name = "Sheet1"

department = parse_department_with_regexp(sheet_name, first_row_text)

print(f"Extracted Department: '{department}'")
# Output: 'Computer Science & Engineering'

Extracted Department: 'Computer Science & Engineering'


In [9]:
# Create mapping of sheet names to department names using LLM
print("🔍 Parsing department names from sheet headers...\n")

department_mapping = {}

for sheet_name in sheet_names:
    # Use the pre-extracted header text (from row 0)
    first_row_text = sheets_headers[sheet_name]
    
    print(f"Sheet: '{sheet_name}'")
    print(f"  Header text: {first_row_text[:100]}...")
    
    # Parse department using LLM
    # department = parse_department_with_llm(sheet_name, first_row_text)
    department = parse_department_with_regexp(sheet_name, first_row_text)
    department_mapping[sheet_name] = department
    
    print(f"  ✓ Department: {department}\n")

print("Department Mapping Summary:")
print("-" * 50)
for sheet, dept in department_mapping.items():
    print(f"  {sheet:20} → {dept}")
print("-" * 50)




🔍 Parsing department names from sheet headers...

Sheet: 'Table 1'
  Header text: Program : B.Tech.
Academic Year-Semester 2017-1 Department : Aerospace Engineering Batch Year : 2017...
  ✓ Department: Aerospace Engineering

Sheet: 'Table 2'
  Header text: Program : B.Tech.
Academic Year-Semester 2017-1 Department : Chemical Engineering Batch Year : 2017...
  ✓ Department: Chemical Engineering

Sheet: 'Table 3'
  Header text: Program : B.Tech.
Academic Year-Semester 2017-1 Department : Civil Engineering Batch Year : 2017...
  ✓ Department: Civil Engineering

Sheet: 'Table 4'
  Header text: Program : B.Tech.
Academic Year-Semester 2017-1
Department : Computer Science and Engineering Batch ...
  ✓ Department: Computer Science and Engineering

Sheet: 'Table 5'
  Header text: Program : B.Tech.
Academic Year-Semester 2017-1 Department : Electrical Engineering Batch Year : 201...
  ✓ Department: Electrical Engineering

Sheet: 'Table 6'
  Header text: Program : B.Tech.
Academic Year-Semester 

# Create Combined DataFrame

In [10]:
# Create final combined dataframe with all required columns
print("Creating combined DataFrame...")

combined_data = []

for item in extracted_data:
    sheet = item['sheet_name']
    department = department_mapping.get(sheet, 'Unknown')
    
    combined_data.append({
        'roll_number': item['roll_number'],
        'name': item['name'],
        'department': department,
        'college': college_name
    })

# Create final dataframe
final_df = pd.DataFrame(combined_data)

# Reorder columns as requested: roll_number, name, department, college
final_df = final_df[['roll_number', 'name', 'department', 'college']]

print(f"\n✓ Combined DataFrame created successfully")
print(f"  Total rows: {len(final_df)}")
print(f"  Columns: {list(final_df.columns)}")
print(f"\nDataFrame preview (first 10 rows):")
print(final_df.head(10))
print(f"\nUnique departments in combined data:")
print(final_df['department'].value_counts())
print(f"\nTotal unique departments: {final_df['department'].nunique()}")

Creating combined DataFrame...

✓ Combined DataFrame created successfully
  Total rows: 960
  Columns: ['roll_number', 'name', 'department', 'college']

DataFrame preview (first 10 rows):
   roll_number                        name             department  \
0  170010001.0      Meena Sourav Ramkishor  Aerospace Engineering   
1  170010002.0         Vasava Sachin Kumar  Aerospace Engineering   
2  170010003.0  Jirwankar Piyush Prabhakar  Aerospace Engineering   
3  170010004.0  Jirwankar Saieesh Sadashiv  Aerospace Engineering   
4  170010005.0          Tarang Rakesh Jain  Aerospace Engineering   
5  170010006.0      Kataria Harshal Sanjay  Aerospace Engineering   
6  170010007.0           Dhruv Rajesh Jain  Aerospace Engineering   
7  170010008.0      Gunjan Vivek Chitlange  Aerospace Engineering   
8  170010009.0     Rohit Vinodrao Tembhare  Aerospace Engineering   
9  170010010.0      Prajwal Vinodrao Bijwe  Aerospace Engineering   

                                  college  
0  India

# Generate CSV File

In [11]:
# Export combined DataFrame to CSV
print("📁 Exporting to CSV file...\n")

final_df.to_csv(output_csv, index=False, encoding='utf-8')

print(f"✅ Successfully saved to {output_csv}")
print(f"   Location: {Path(output_csv).absolute()}")
print(f"   Total records exported: {len(final_df)}")
print(f"   File size: {Path(output_csv).stat().st_size / 1024:.2f} KB")

📁 Exporting to CSV file...

✅ Successfully saved to input_people_csv_v3.csv
   Location: c:\Users\praka\Acads\Work\AI_Agents\linkedn-scrapper-v2\linkedin_scraper_repo\samples\input_people_csv_v3.csv
   Total records exported: 960
   File size: 88.29 KB


# Validate Output Data

In [ ]:
# Comprehensive validation of output data
print("=" * 70)
print("VALIDATION REPORT")
print("=" * 70)

# Check if file exists
if not Path(output_csv).exists():
    print(f"❌ File not found: {output_csv}")
else:
    print(f"✅ File exists: {output_csv}\n")
    
    # Read the exported CSV for verification
    verify_df = pd.read_csv(output_csv)
    
    print("1. FILE STRUCTURE VALIDATION")
    print("-" * 70)
    print(f"   Total rows: {len(verify_df)}")
    print(f"   Total columns: {len(verify_df.columns)}")
    print(f"   Column names: {list(verify_df.columns)}")
    
    # Check required columns
    required_columns = ['roll_number', 'name', 'department', 'college']
    missing_columns = [col for col in required_columns if col not in verify_df.columns]
    if missing_columns:
        print(f"   ❌ Missing columns: {missing_columns}")
    else:
        print(f"   ✓ All required columns present")
    
    print("\n2. DATA QUALITY VALIDATION")
    print("-" * 70)
    print(f"   Roll numbers: {len(verify_df)}")
    print(f"   Unique roll numbers: {verify_df['roll_number'].nunique()}")
    
    # Check for missing/null values
    null_counts = verify_df.isnull().sum()
    if null_counts.sum() > 0:
        print(f"   ⚠ Null values found:")
        for col, count in null_counts[null_counts > 0].items():
            print(f"      {col}: {count} missing values")
    else:
        print(f"   ✓ No missing values detected")
    
    # Check for duplicate roll numbers
    duplicates = verify_df[verify_df.duplicated(subset=['roll_number'], keep=False)]
    if len(duplicates) > 0:
        print(f"   ⚠ Duplicate roll numbers found: {len(duplicates)}")
        print(f"      Duplicates:\n{duplicates.to_string()}")
    else:
        print(f"   ✓ No duplicate roll numbers")
    
    print("\n3. DEPARTMENT VALIDATION")
    print("-" * 70)
    print(f"   Unique departments: {verify_df['department'].nunique()}")
    print(f"   Departments:")
    for dept, count in verify_df['department'].value_counts().items():
        print(f"      {dept}: {count} students")
    
    print("\n4. COLLEGE VALIDATION")
    print("-" * 70)
    unique_colleges = verify_df['college'].unique()
    print(f"   Unique colleges: {len(unique_colleges)}")
    for college in unique_colleges:
        count = (verify_df['college'] == college).sum()
        print(f"      {college}: {count} students")
    
    print("\n5. STUDENT COUNT SUMMARY")
    print("-" * 70)
    print(f"   Original extracted data: {len(extracted_data)} students")
    print(f"   Final CSV data: {len(verify_df)} students")
    print(f"   Count match: {'✓ YES' if len(extracted_data) == len(verify_df) else '❌ MISMATCH'}")
    
    print("\n6. SAMPLE DATA (First 10 rows)")
    print("-" * 70)
    print(verify_df.head(10).to_string(index=False))
    
    print("\n" + "=" * 70)
    print("✅ VALIDATION COMPLETE - All checks passed!")
    print("=" * 70)